# Evaluation: the accuracy gate

The dataset is famously separable, so the gate is strict: **accuracy ≥ 0.9**
through the exact artifact the pipeline would register.

In [ ]:
import pathlib
import tarfile
import tempfile
from io import BytesIO

import boto3
import joblib
import pandas as pd
from feast import FeatureStore
from sklearn.metrics import accuracy_score, confusion_matrix
from wine_quality_classification.feature_contract import (
    FEATURE_COLUMNS,
    LABEL_COLUMN,
)

BUCKET = "wine-quality-classification"
s3r = boto3.resource(
    "s3", endpoint_url="http://wine-quality-classification-moto:5000"
)
jobs = sorted(
    o.key
    for o in s3r.Bucket(BUCKET).objects.filter(Prefix="output/")
    if o.key.endswith("model.tar.gz")
)
buf = BytesIO()
s3r.Bucket(BUCKET).download_fileobj(jobs[-1], buf)
buf.seek(0)
tmp = pathlib.Path(tempfile.mkdtemp())
with tarfile.open(fileobj=buf) as t:
    t.extractall(tmp)
model = joblib.load(tmp / "model.joblib")

store = FeatureStore(repo_path="feature_repo")
entities = pd.read_parquet("data/processed/wine_labels.parquet")
df = store.get_historical_features(
    entity_df=entities,
    features=[f"wine_assay_profile:{c}" for c in FEATURE_COLUMNS]
    + [f"wine_cultivar_labels:{LABEL_COLUMN}"],
).to_df()
preds = model.predict(df[FEATURE_COLUMNS])
acc = float(accuracy_score(df[LABEL_COLUMN], preds))
acc

In [ ]:
assert acc >= 0.9, f"accuracy gate failed: {acc}"
"gate passed"

In [ ]:
import matplotlib.pyplot as plt

cm = confusion_matrix(df[LABEL_COLUMN], preds)
plt.imshow(cm, cmap="Blues")
plt.xticks(range(3))
plt.yticks(range(3))
plt.xlabel("predicted")
plt.ylabel("actual")
plt.colorbar()